# Practical Guide: Introduction to LangChain

> **BIA® School of Technology & AI — Generative AI & Agentic AI Development**  
> Pre-requisite reading for Session 7: Structured Prompting

LangChain is the industry-standard Python framework for building applications with LLMs.  
Instead of writing raw API calls, you work with **composable building blocks** that snap together.

**What this guide covers:**
1. Installation & setup
2. Your first LLM call with `ChatOpenAI`
3. Prompt templates (`ChatPromptTemplate`)
4. LCEL — the pipe (`|`) operator
5. Output parsers (text, JSON, Pydantic)
6. Structured output with `with_structured_output()`
7. Tools & tool calling
8. Conversational memory
9. Document loading & text splitting (RAG preview)
10. Putting it all together

**Estimated time:** 45–60 minutes

---

## 1. Installation & Setup

LangChain is split into focused packages. You install only what you need:

| Package | What it provides |
|---------|------------------|
| `langchain-core` | Base classes: prompts, parsers, runnables, messages |
| `langchain-openai` | OpenAI integration (`ChatOpenAI`, embeddings) |
| `langchain` | Chains, agents, memory, document loaders |
| `langgraph` | Graph-based agent orchestration (production agents) |
| `langchain-community` | 100+ third-party integrations |

**Current versions (March 2026):** langchain-core 1.2.x, langchain-openai 1.1.x, langgraph 1.1.x

In [ ]:
# Run once to install
# !pip install langchain-openai langchain-core langchain langgraph pydantic

In [ ]:
import os

# Set your OpenAI API key (choose one method):
# Method 1: Environment variable (recommended)
#   In your terminal:  export OPENAI_API_KEY="sk-..."

# Method 2: Set it here (for quick testing only)
# os.environ["OPENAI_API_KEY"] = "sk-YOUR-KEY-HERE"

# Verify it's set
assert os.environ.get("OPENAI_API_KEY"), "Please set OPENAI_API_KEY first!"
print("API key is set.")

---
## 2. Your First LLM Call — `ChatOpenAI`

`ChatOpenAI` is LangChain's wrapper around OpenAI's chat models.  
It replaces the raw `openai.OpenAI()` client with a **composable** object.

Key differences from the raw SDK:
- `.invoke()` instead of `client.chat.completions.create()`
- Returns an `AIMessage` object (not a raw dict)
- Can be piped with `|` into prompts, parsers, and other components

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the model
llm = ChatOpenAI(
    model="gpt-4o-mini",      # The model to use
    temperature=0.0,           # 0 = deterministic, 1 = creative
    max_tokens=500,            # Maximum response length
)

# Simple string input → the model treats it as a human message
response = llm.invoke("What is LangChain in one sentence?")

print(type(response))          # <class 'langchain_core.messages.ai.AIMessage'>
print(response.content)        # The actual text

In [ ]:
# You can also pass a list of messages (system + human)
from langchain_core.messages import SystemMessage, HumanMessage

response = llm.invoke([
    SystemMessage(content="You are a helpful Python tutor. Be concise."),
    HumanMessage(content="What's the difference between a list and a tuple?"),
])

print(response.content)

In [ ]:
# Shorthand: tuples instead of message objects
# ("system", "...") is equivalent to SystemMessage(content="...")

response = llm.invoke([
    ("system", "You are a helpful Python tutor. Be concise."),
    ("human", "What's a list comprehension?"),
])

print(response.content)

**Key point:** `llm.invoke()` is the universal method. Every LangChain component has `.invoke()` — prompts, models, parsers, chains, agents. This consistency is what makes them composable.

---
## 3. Prompt Templates — `ChatPromptTemplate`

Hardcoding prompts in strings gets messy fast. `ChatPromptTemplate` gives you **reusable prompts with variables**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a template with {placeholders}
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {topic}. Explain concepts clearly for beginners."),
    ("human", "{question}"),
])

# .invoke() fills in the placeholders and returns formatted messages
formatted = prompt.invoke({"topic": "machine learning", "question": "What is overfitting?"})

print("Formatted messages:")
for msg in formatted.messages:
    print(f"  [{msg.__class__.__name__}] {msg.content}")

In [ ]:
# You can invoke the formatted prompt with the model directly
response = llm.invoke(formatted)
print(response.content)

But there's a better way — **LCEL chains**.

---
## 4. LCEL — The Pipe Operator

**LCEL** = LangChain Expression Language. It's the `|` operator that connects components:

```python
chain = prompt | model | parser
```

Data flows left to right:
1. `prompt.invoke({...})` → formatted messages
2. `model.invoke(messages)` → AIMessage
3. `parser.invoke(ai_message)` → final output (string, dict, Pydantic object)

You call `.invoke()` on the **chain**, and it runs all three steps automatically.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Build an LCEL chain: prompt → model → string parser
explain_chain = prompt | llm | StrOutputParser()

# One call runs the entire pipeline
result = explain_chain.invoke({
    "topic": "databases",
    "question": "What is the difference between SQL and NoSQL?"
})

print(type(result))  # <class 'str'> — not AIMessage, thanks to StrOutputParser
print(result)

In [ ]:
# Reuse the same chain with different inputs
print(explain_chain.invoke({"topic": "networking", "question": "What is DNS?"}))
print("\n" + "="*50 + "\n")
print(explain_chain.invoke({"topic": "security", "question": "What is encryption?"}))

In [ ]:
# LCEL also supports streaming (token by token)
for chunk in explain_chain.stream({"topic": "Python", "question": "What are decorators?"}):
    print(chunk, end="", flush=True)

**Why LCEL matters:**
- **Reusable** — define once, call `.invoke()` with any inputs
- **Composable** — swap the model without changing the prompt or parser
- **Streaming** — `.stream()` works automatically on any chain
- **Async** — `.ainvoke()` and `.astream()` for async Python
- **Batch** — `.batch([input1, input2, ...])` processes multiple inputs

---
## 5. Output Parsers

Output parsers transform the model's raw response into the format you need.

| Parser | Output | Use Case |
|--------|--------|----------|
| `StrOutputParser()` | `str` | General text responses |
| `JsonOutputParser()` | `dict` | JSON data extraction |
| `PydanticOutputParser(pydantic_object=...)` | Pydantic model | Typed, validated objects |

In [ ]:
# ── StrOutputParser: returns plain text ───────────────────────────
from langchain_core.output_parsers import StrOutputParser

text_chain = (
    ChatPromptTemplate.from_messages([
        ("human", "Give me a one-line summary of {concept}")
    ])
    | llm
    | StrOutputParser()
)

result = text_chain.invoke({"concept": "microservices"})
print(type(result))  # str
print(result)

In [ ]:
# ── JsonOutputParser: returns a Python dict ──────────────────────
from langchain_core.output_parsers import JsonOutputParser

json_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Always respond with valid JSON."),
        ("human", "List 3 {language} frameworks as JSON: "
                  '{{"frameworks": [{{"name": "...", "use_case": "..."}}]}}')
    ])
    | llm
    | JsonOutputParser()
)

result = json_chain.invoke({"language": "Python"})
print(type(result))  # dict
for fw in result["frameworks"]:
    print(f"  {fw['name']}: {fw['use_case']}")

---
## 6. Structured Output — The Modern Way

`with_structured_output()` is the **recommended** approach for getting typed data from LLMs.  
You define a Pydantic model (a Python class with typed fields), and LangChain ensures the model returns data matching that schema.

**This is better than JsonOutputParser because:**
- No prompt engineering needed to get JSON — the schema is sent to the model natively
- Pydantic validates the response — wrong types raise errors immediately
- You get Python objects with dot notation (`result.name`), not dicts (`result["name"]`)

In [ ]:
from pydantic import BaseModel, Field
from typing import List


# Step 1: Define your schema as a Pydantic model
class MovieRecommendation(BaseModel):
    """A movie recommendation with details."""
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    genre: str = Field(description="Primary genre")
    reason: str = Field(description="Why this movie is recommended")


class MovieList(BaseModel):
    """A list of movie recommendations."""
    movies: List[MovieRecommendation] = Field(description="List of recommended movies")


# Step 2: Bind the schema to the model
structured_llm = llm.with_structured_output(MovieList)

# Step 3: Use it in a chain
movie_chain = (
    ChatPromptTemplate.from_messages([
        ("human", "Recommend 3 {genre} movies for someone who liked {reference_movie}")
    ])
    | structured_llm
)

result = movie_chain.invoke({"genre": "sci-fi", "reference_movie": "Inception"})

# result is a MovieList object — access with dot notation!
print(type(result))  # <class 'MovieList'>
for movie in result.movies:
    print(f"  {movie.title} ({movie.year}) — {movie.genre}")
    print(f"    {movie.reason}")

---
## 7. Tools & Tool Calling

**Tools** let the model call Python functions. This is the foundation of AI agents.

LangChain provides two ways to work with tools:
1. `@tool` decorator — define tools as Python functions
2. `llm.bind_tools()` — tell the model which tools it can call
3. `create_react_agent()` — full agent loop that calls tools automatically

In [ ]:
from langchain_core.tools import tool
import math


# Define tools with the @tool decorator
# The decorator reads: function name, docstring, type hints

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Returns temperature and conditions."""
    # In production, this would call a real weather API
    weather_data = {
        "pune": "32°C, Sunny",
        "mumbai": "30°C, Humid",
        "delhi": "28°C, Partly Cloudy",
        "bangalore": "25°C, Pleasant",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")


@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression. Example: '25 * 4 + 10'"""
    try:
        result = eval(expression, {"__builtins__": {}},
                      {"abs": abs, "round": round, "sqrt": math.sqrt})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# Inspect the tool metadata (auto-generated from the decorator)
tools = [get_weather, calculate]
for t in tools:
    print(f"{t.name}: {t.description}")
    print(f"  Args schema: {t.args}\n")

In [ ]:
# ── bind_tools(): Tell the model about available tools ─────────────
# The model doesn't call tools directly — it returns a tool_calls list
# saying WHICH tool to call with WHAT arguments.

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What's the weather in Pune?")

print("Content:", response.content)        # Usually empty when tool is called
print("Tool calls:", response.tool_calls)   # Structured tool call request

# The model says: "Call get_weather with city='Pune'"
# It's YOUR job (or an agent's job) to actually execute it.

In [ ]:
# ── Manually execute a tool call ─────────────────────────────────
if response.tool_calls:
    tc = response.tool_calls[0]
    tool_name = tc["name"]
    tool_args = tc["args"]

    # Find the tool and run it
    tool_map = {t.name: t for t in tools}
    result = tool_map[tool_name].invoke(tool_args)

    print(f"Called: {tool_name}({tool_args})")
    print(f"Result: {result}")

In [ ]:
# ── create_react_agent(): Automate the full loop ─────────────────
# Instead of manually executing tool calls, LangGraph handles it.

from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant. Use tools when needed.",
)

# The agent handles: model call → tool execution → feed result back → repeat
result = agent.invoke({
    "messages": [{"role": "user", "content": "What's the weather in Pune and Mumbai?"}]
})

# Print the full conversation
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "content") and msg.content:
        print(f"[{role}] {msg.content[:200]}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  -> Calls: {tc['name']}({tc['args']})")

**Tool calling flow:**

```
User question
     ↓
Model decides: "I need to call get_weather(city='Pune')"
     ↓
Agent executes the tool → gets "32°C, Sunny"
     ↓
Model receives result, decides: do I need more tools?
     ↓
No → Model generates final answer to user
```

This is the **ReAct pattern** — Reason (decide what to do) + Act (call the tool). We'll go deeper in Session 7.

---
## 8. Conversational Memory

By default, each `.invoke()` call is **stateless** — the model doesn't remember previous messages.  
To build a chatbot, you need to manage conversation history.

The modern LangChain approach uses `RunnableWithMessageHistory`.

In [ ]:
# ── Simple approach: manage history yourself ─────────────────────
from langchain_core.messages import HumanMessage, AIMessage

# Start a conversation
history = [
    ("system", "You are a helpful cooking assistant. Be concise."),
]

def chat(user_message: str) -> str:
    """Send a message and get a response, maintaining history."""
    history.append(("human", user_message))
    response = llm.invoke(history)
    history.append(("assistant", response.content))
    return response.content


# Multi-turn conversation
print("User: I want to make pasta")
print("AI:", chat("I want to make pasta"))
print()
print("User: What ingredients do I need?")
print("AI:", chat("What ingredients do I need?"))  # It remembers we're talking about pasta!
print()
print("User: Make it vegetarian")
print("AI:", chat("Make it vegetarian"))  # It remembers the full context

In [ ]:
# ── LangChain approach: RunnableWithMessageHistory ───────────────
# This wraps any chain to automatically manage conversation history
# per session (useful for multi-user apps).

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Store for conversation histories (keyed by session ID)
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# Create a chain with memory
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("placeholder", "{history}"),
    ("human", "{input}"),
])

chain = chat_prompt | llm | StrOutputParser()

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# Use it with a session ID
config = {"configurable": {"session_id": "user-123"}}

print(chain_with_memory.invoke({"input": "My name is Danny"}, config=config))
print()
print(chain_with_memory.invoke({"input": "What's my name?"}, config=config))

---
## 9. Document Loading & Text Splitting (RAG Preview)

LangChain provides tools to load documents from many sources and split them into chunks for processing.  
This is the foundation of **RAG** (Retrieval-Augmented Generation) — we'll cover it fully in later sessions.

Here's a quick taste:

In [ ]:
# ── Document Loaders ─────────────────────────────────────────────
# LangChain has 100+ loaders: PDF, CSV, HTML, Notion, Slack, etc.
# Here we use the simplest one: loading from a string.

from langchain_core.documents import Document

# In production you'd use:
#   from langchain_community.document_loaders import PyPDFLoader
#   docs = PyPDFLoader("report.pdf").load()

# For this demo, create documents manually
docs = [
    Document(
        page_content="LangChain is a framework for building LLM applications. "
                     "It provides tools for prompt management, chains, agents, "
                     "and memory. The latest version uses LCEL for composability.",
        metadata={"source": "langchain-docs", "page": 1}
    ),
    Document(
        page_content="LangGraph extends LangChain for building stateful, multi-actor "
                     "applications with LLMs. It models agent workflows as graphs "
                     "with nodes (steps) and edges (transitions).",
        metadata={"source": "langgraph-docs", "page": 1}
    ),
]

for doc in docs:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content[:80]}...\n")

In [ ]:
# ── Text Splitting ───────────────────────────────────────────────
# Large documents need to be split into chunks that fit the model's
# context window. RecursiveCharacterTextSplitter is the go-to choice.

from langchain_text_splitters import RecursiveCharacterTextSplitter

long_text = """Artificial intelligence has transformed how we build software. 
Modern AI applications use large language models as their core reasoning engine. 
These models can understand natural language, generate code, and make decisions. 
Frameworks like LangChain make it easier to build production AI applications 
by providing reusable components for common patterns like RAG, agents, and chains. 
The key challenge is managing context — models have limited context windows, 
so we need to carefully select what information to include in each prompt."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,     # Max characters per chunk
    chunk_overlap=30,   # Overlap between chunks (preserves context)
)

chunks = splitter.split_text(long_text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk[:60]}...")

---
## 10. Putting It All Together

Let's build a small but complete application that uses **everything** we've learned:  
prompt templates, LCEL chains, structured output, and tool calling.

In [ ]:
# ── Mini Project: Code Review Assistant ──────────────────────────
# Given a code snippet, the assistant:
# 1. Analyzes it using structured output (Pydantic)
# 2. Returns typed, validated feedback

class CodeIssue(BaseModel):
    """A single issue found in code."""
    line_hint: str = Field(description="Which part of the code has the issue")
    severity: str = Field(description="low, medium, or high")
    issue: str = Field(description="What the problem is")
    fix: str = Field(description="How to fix it")


class CodeReview(BaseModel):
    """Complete code review with issues and overall assessment."""
    language: str = Field(description="Programming language detected")
    issues: List[CodeIssue] = Field(description="List of issues found")
    overall_quality: str = Field(description="good, needs_improvement, or poor")
    summary: str = Field(description="One-sentence summary of the review")


# Build the chain
review_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a senior code reviewer. Analyze the given code "
                   "for bugs, style issues, and potential improvements."),
        ("human", "Review this code:\n\n```\n{code}\n```"),
    ])
    | llm.with_structured_output(CodeReview)
)

# Test it
review = review_chain.invoke({"code": """
def calculate_average(numbers):
    total = 0
    for i in range(len(numbers)):
        total = total + numbers[i]
    average = total / len(numbers)
    return average
"""})

print(f"Language: {review.language}")
print(f"Quality: {review.overall_quality}")
print(f"Summary: {review.summary}")
print(f"\nIssues ({len(review.issues)}):")
for issue in review.issues:
    print(f"  [{issue.severity.upper()}] {issue.issue}")
    print(f"    Fix: {issue.fix}")

---
## Quick Reference

| Task | Code |
|------|------|
| Initialize model | `llm = ChatOpenAI(model="gpt-4o-mini")` |
| Simple call | `llm.invoke("Hello")` |
| With messages | `llm.invoke([("system", "..."), ("human", "...")])` |
| Prompt template | `ChatPromptTemplate.from_messages([("system", "..."), ("human", "{var}")])` |
| LCEL chain | `chain = prompt \| llm \| StrOutputParser()` |
| Run a chain | `chain.invoke({"var": "value"})` |
| Stream a chain | `for chunk in chain.stream({...}): print(chunk)` |
| Structured output | `llm.with_structured_output(MyPydanticModel)` |
| Define a tool | `@tool` decorator with docstring + type hints |
| Bind tools | `llm.bind_tools([tool1, tool2])` |
| ReAct agent | `create_react_agent(model=llm, tools=tools)` |
| Memory | `RunnableWithMessageHistory(chain, get_history_fn)` |

**Key packages:**

```
pip install langchain-openai langchain-core langchain langgraph
```

**Key imports:**

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field
```

---

**Next:** Session 7 notebook applies these concepts to structured prompting patterns (CoT, ReAct, ToT, GoT).

---
*© BIA® School of Technology & AI — Generative AI & Agentic AI Development Program*